In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
# 暫時禁用future warningq
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
# 專案根目錄（alphagen-master/alphagen-master）
# Jupyter cwd 在 testing/ 時用 parent；否則用 cwd
_cwd = Path.cwd()
root = _cwd.parent if _cwd.name == "testing" else _cwd
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Project root: {root}")

In [ ]:
from src.alphas.alpha191 import Alpha191
alphas  = Alpha191.get_all_factors()
out_dir = '/home/newt/quant_project/src/factor/test_result/alpha191'
# alpha_str = Alpha101.alpha013()  


In [ ]:
from src.factor_engine import FactorEngine

RAM_PATH="/dev/shm/qlib_ram"     # 改成你的實際路徑

if globals().get("engine", None) is None:
    engine = FactorEngine(
        qlib_data_path=RAM_PATH,
        instrument="all",
        start_time="2020-01-01",
        end_time="2026-01-20",
        device="cpu",               # 支援 "cpu" / "cuda:0" 字串或 torch.device
        max_backtrack_days=300,
        max_future_days=0,
        adjust_factor = 'back'
    )
if globals().get("df_returns", None) is None:
    df_close = engine.calc_df("$close", col_name="close").sort_index()

    # 2) 要算的區間：(名稱, 賣出日相對 T+1 的偏移)
    #    例如 1day_return = (T+2)/(T+1)-1 → lead 2 和 lead 1
    return_windows = [
        ("1day_return", 2, 1),   # close[T+2]/close[T+1] - 1
        ("3day_return", 4, 1),   # close[T+4]/close[T+1] - 1
        ("5day_return", 6, 1),
        ("10day_return", 11, 1),
        ("20day_return", 21, 1),
        ("30day_return", 31, 1),
    ]

    out = []
    for name, end_lead, start_lead in return_windows:
        s = (
            df_close.groupby(level="instrument", group_keys=False)["close"]
            .transform(lambda x: x.shift(-end_lead) / x.shift(-start_lead) - 1)
        )
        s.name = name
        out.append(s)

    df_returns = pd.concat(out, axis=1)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def process_untradable_and_disposed(
    df: pd.DataFrame,
    drop: bool = False,
    keep_flag: bool = False,
    show_daily_na: bool = True,
) -> pd.DataFrame:
    """
    融合處理：先將不可交易/已處置的列在因子欄位設為 NaN，再可選顯示每日 NA 比例，最後回傳與原邏輯相同。
    流程：篩掉不可交易 → 查看 NA 比例（show_daily_na）→ 可選 drop Alpha NA → 回傳。
    """
    if "$is_tradable" not in df.columns or "$is_disposed" not in df.columns:
        raise ValueError("DataFrame 中缺少 $is_tradable 或 $is_disposed 欄位")

    # valid = (df["$is_tradable"] == 1) & (df["$is_disposed"] == 0)
    # factor_cols = [c for c in df.columns if c not in ("$is_tradable", "$is_disposed")]
    # merged_masked = df.copy()
    # merged_masked.loc[~valid, factor_cols] = np.nan
    valid = (df["$is_tradable"] == 1) & (df ["$is_disposed"] == 0)
    factor_cols = [c for c in df.columns if c not in ("$is_tradable", "$is_disposed")]
    merged_masked = df.loc[valid].copy()
    if show_daily_na:

        if isinstance(merged_masked.index, pd.MultiIndex):
            dates = merged_masked.index.get_level_values(0)
        elif "date" in merged_masked.columns:
            dates = merged_masked["date"]
        else:
            raise ValueError("df 需有 date 欄位或 (date, instrument) MultiIndex")
        rows = []
        for date, g in merged_masked.groupby(dates):
            n = len(g)
            row = {"date": date, "總筆數": n}
            for col in factor_cols:
                if col not in g.columns:
                    continue
                na_count = g[col].isna().sum()
                pct = (na_count / n * 100) if n else 0.0
                row[f"{col}_NA數"] = na_count
                row[f"{col}_NA_ratio"] = pct
            rows.append(row)
        stats = pd.DataFrame(rows)

        # 用比例做分箱，Y 軸為次數（有幾天落在該比例區間）
        bins = np.linspace(0, 100, 21)  # 0, 5, 10, ..., 100
        n_factors = len([c for c in factor_cols if f"{c}_NA_ratio" in stats.columns])
        fig, axes = plt.subplots(n_factors, 1, figsize=(8, 4 * n_factors))
        if n_factors == 1:
            axes = [axes]
        for ax, col in enumerate(factor_cols):
            if f"{col}_NA_ratio" not in stats.columns:
                continue
            ratio = stats[f"{col}_NA_ratio"]
            counts, bin_edges = np.histogram(ratio, bins=bins)
            bin_labels = [f"{bin_edges[i]:.0f}–{bin_edges[i+1]:.0f}%" for i in range(len(bin_edges)-1)]
            axes[ax].bar(bin_labels, counts, width=0.8, edgecolor="gray")
            axes[ax].set_xlabel("NA ratio")
            axes[ax].set_ylabel("Times")
            axes[ax].set_title(f"{col} daily NA ratio dist")
            axes[ax].tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()

    if "Alpha" in merged_masked.columns and drop:
        merged_masked = merged_masked.dropna(subset=["Alpha"])

    if keep_flag:
        return merged_masked
    return merged_masked[factor_cols]
def get_tradable_disposed_mask(df: pd.DataFrame) -> pd.Series:
    """
    回傳「可交易且未處置」的布林 mask（與 df 同 index）。
    可用 df.loc[mask] 直接篩選，避免重複計算。
    """
    if "$is_tradable" not in df.columns or "$is_disposed" not in df.columns:
        raise ValueError("DataFrame 中缺少 $is_tradable 或 $is_disposed 欄位")
    return (df["$is_tradable"] == 1) & (df["$is_disposed"] == 0)
def apply_tradable_mask(df, mask):
    m = mask.reindex(df.index, fill_value=False)
    return df.loc[m]
df_is_tradable = engine.load_raw_df(['$is_tradable' , '$is_disposed'])

tradable_mask = get_tradable_disposed_mask(df_is_tradable)

In [6]:
import numpy as np
import pandas as pd
from panda_factor.panda_factor.panda_factor.analysis.factor_func import grouping_factor, cal_pct_lag , ext_out_3std , ext_out_mad
from panda_factor.panda_factor.panda_factor.analysis.factor import factor
import base64
import io
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib
import os
import matplotlib.pyplot as plt
def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

def df_to_plotly_table(df, title):
    return go.Figure(data=[go.Table(
        header=dict(values=[""] + list(df.columns), fill_color="lightblue"),
        cells=dict(
            values=[df.index.tolist()] + [df[c].astype(str).tolist() for c in df.columns],
            fill_color="white",
        ),
    )], layout=dict(title=title, height=200 + 25 * len(df)))
for alpha_name, alpha_str in alphas.items():
    df = pd.DataFrame()
    factor_name = 'Alpha'
    # 檢查去極值後的欄位inf 狀態
    alpha_factors = {
        "Alpha": alpha_str,
    }
    name_list = list(alpha_factors.keys())
    exprs_list = list(alpha_factors.values())
    df = engine.calc_many_df(expr_strs=exprs_list , names=name_list)
    if factor_name in df.columns:
        inf_count = np.isinf(df[factor_name]).sum()
        #print(f"{factor_name} 中 inf 數量: {inf_count}")
        if inf_count > 0:
            pass
            #print(df.loc[np.isinf(df[factor_name]), [factor_name]].head())
    # 假設您的 qlib_df index 是 (datetime, instrument)
    # 1. 轉換 index 到 columns
    df = apply_tradable_mask(df, tradable_mask)
    df = df.dropna(subset=["Alpha"])
    df_factor = pd.merge(df, df_returns, on=["datetime", "instrument"], how="left")

    # 2. 重命名欄位以符合 PandaFactor 習慣 (非強制，但建議)
    # 確保 DataFrame 結構為：[date, symbol, factor_value]
    rename_map = {
        'datetime': 'date',
        'instrument': 'symbol',
    }
    df_factor.reset_index(inplace=True)
    df_factor = df_factor.rename(columns=rename_map)
    # 3. 確保日期格式為字串或與系統一致 (factor_analysis 會處理 pd.to_datetime)
    df_factor['date'] = pd.to_datetime(df_factor['date'])


    # 假設你已經有合併好的 df（包含因子和 K 線數據）
    # 但還沒有分組列
    # 1. 計算滯後收益（如果還沒有）
    if 'returns_lag1' not in df_factor.columns:
        df_factor = cal_pct_lag(df_factor)

    df_factor = df_factor.groupby('date', group_keys=False).apply(
        lambda g: ext_out_3std(g.copy(), factor_name , process_inf=True)
    )
    # 2. 進行分組（這一步會添加 'Alpha012_group' 列）
    # grouping_factor 內對因子做 dropna()，若當日有 NA 會導致分組結果長度與當日列數不一致；先只保留因子有效列再分組
    df_factor_for_grouping = df_factor[np.isfinite(df_factor[factor_name])].copy()
    df_cuted, df_benchmark = grouping_factor(
        df=df_factor_for_grouping,
        factor_name=factor_name,
        group_cnt=5
    )

    alpha = factor(name = factor_name , group_number = 5 )
    alpha.set_backtest_parameters(
        period = 1,
        predict_direction=1
    )
    alpha.start_backtest(df_cuted, df_benchmark)

    # 先檢查 ic_info：IC_mean 或 Rank_IC 任一超過 0.02 且 t统计量 > 2 才畫圖
    ic_info = alpha.df_info2
    ic_mean = float(ic_info.loc['IC_mean', alpha.name])
    rank_ic = float(ic_info.loc['Rank_IC', alpha.name])
    t_stat = float(ic_info.loc['t统计量', alpha.name])
    if not ((ic_mean > 0.02 or rank_ic > 0.02) and t_stat > 2):
        print(f"{alpha_name}: IC_mean={ic_mean}, Rank_IC={rank_ic}, t统计量={t_stat} 未滿足（任一>0.02 且 t>2），略過畫圖")
        continue

    matplotlib.use("Agg")

    


    # 暫時不 show
    _orig_show = plt.show
    plt.show = lambda: None

    # 先畫出四張圖並轉成 base64
    alpha.draw_pct()
    img_pct = fig_to_base64(plt.gcf())
    plt.close()
    alpha.draw_ic(0)
    img_ic0 = fig_to_base64(plt.gcf())
    plt.close()
    alpha.draw_ic(1)
    img_ic1 = fig_to_base64(plt.gcf())
    plt.close()
    alpha.draw_ic_dacay()
    img_decay = fig_to_base64(plt.gcf())
    plt.close()
    plt.show = _orig_show

    # 組 HTML：表用 Plotly Table（或可改成 df.to_html()），圖用 <img src="data:image/png;base64,...">
    fig_t1 = df_to_plotly_table(alpha.df_info, f"{alpha_name} 多空分組統計")
    fig_t2 = df_to_plotly_table(alpha.df_info2, f"{alpha_name} IC 統計")

    html_parts = [
        "<!DOCTYPE html><html><head><meta charset='utf-8'></head><body>",
        f"<h1>{alpha_name} 因子報告</h1>",
        fig_t1.to_html(full_html=False, include_plotlyjs="cdn"),
        fig_t2.to_html(full_html=False, include_plotlyjs=False),
        "<h2>分組收益</h2><img src='data:image/png;base64," + img_pct + "' style='max-width:100%'>",
        "<h2>IC 時序 (normal)</h2><img src='data:image/png;base64," + img_ic0 + "' style='max-width:100%'>",
        "<h2>IC 時序 (rank)</h2><img src='data:image/png;base64," + img_ic1 + "' style='max-width:100%'>",
        "<h2>IC 衰減</h2><img src='data:image/png;base64," + img_decay + "' style='max-width:100%'>",
        "</body></html>",
    ]

    html_path = os.path.join(out_dir, f"{alpha_name}_因子報告.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("".join(html_parts))
    print(f"已儲存: {html_path}")                   

alpha038: IC_mean=0.0027, Rank_IC=0.0148, t统计量=1.4962 未滿足（任一>0.02 且 t>2），略過畫圖
alpha039: IC_mean=0.0068, Rank_IC=0.0157, t统计量=3.3476 未滿足（任一>0.02 且 t>2），略過畫圖
alpha040: IC_mean=0.0027, Rank_IC=-0.0197, t统计量=1.2947 未滿足（任一>0.02 且 t>2），略過畫圖
alpha041: IC_mean=0.0009, Rank_IC=-0.0041, t统计量=0.3354 未滿足（任一>0.02 且 t>2），略過畫圖
alpha042: IC_mean=0.0005, Rank_IC=0.036, t统计量=0.1769 未滿足（任一>0.02 且 t>2），略過畫圖
alpha043: IC_mean=0.0, Rank_IC=-0.0213, t统计量=0.0194 未滿足（任一>0.02 且 t>2），略過畫圖
alpha044: IC_mean=0.0025, Rank_IC=0.0105, t统计量=1.7421 未滿足（任一>0.02 且 t>2），略過畫圖
alpha045: IC_mean=-0.0022, Rank_IC=-0.015, t统计量=-1.1351 未滿足（任一>0.02 且 t>2），略過畫圖
已儲存: /home/newt/quant_project/src/factor/test_result/alpha191/alpha046_因子報告.html
alpha047: IC_mean=-0.002, Rank_IC=-0.011, t统计量=-1.6297 未滿足（任一>0.02 且 t>2），略過畫圖
alpha048: IC_mean=0.0037, Rank_IC=0.0131, t统计量=1.9858 未滿足（任一>0.02 且 t>2），略過畫圖
alpha049: IC_mean=0.0, Rank_IC=0.0133, t统计量=-0.0024 未滿足（任一>0.02 且 t>2），略過畫圖
alpha050: IC_mean=0.0, Rank_IC=-0.0132, t统计量=0.0025 未滿足（任一>0.